# 🦠 COVID-19 Data Cleaning, Visualization & Machine Learning

This notebook contains an end-to-end data science pipeline analyzing global COVID-19 dataset:
1. **Data Cleaning & Handling Missing Values**
2. **Feature Engineering & Rate Calculations**
3. **Exploratory Data Analysis & Visual Dashboards**
4. **Machine Learning (Risk Category Classification)**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker

print('Libraries imported successfully!')

## 1. Data Ingestion & Preprocessing

In [ ]:
# Load dataset
df = pd.read_csv('covid_19.csv')

# Handle missing values
df['continent'] = df['continent'].fillna('CruiseShip')
df['Deaths'] = df['Deaths'].fillna(0)
df['Recovered'] = df['Recovered'].fillna(0)
df['Tests'] = df['Tests'].fillna(0)
df['population'] = df['population'].fillna(0)

# Convert dates
df['day'] = pd.to_datetime(df['day'])
df['time'] = pd.to_datetime(df['time'])

# Filter clean country dataset (excluding double-counted summary rows)
df_clean = df[(df['country'] != 'All') & (df['country'] != df['continent'])].copy()

# Feature engineering
df_clean['Active_Cases'] = df_clean['Cases'] - (df_clean['Recovered'] + df_clean['Deaths'])
df_clean['Death_Rate (%)'] = np.where(df_clean['Cases'] > 0, (df_clean['Deaths'] / df_clean['Cases']) * 100, 0).round(2)
df_clean['Recovery_Rate (%)'] = np.where(df_clean['Cases'] > 0, (df_clean['Recovered'] / df_clean['Cases']) * 100, 0).round(2)

print("Remaining missing values:", df_clean.isnull().sum().sum())
df_clean.head()

## 2. Visual Dashboard of Key Findings

In [ ]:
# Create a 2x2 Grid Dashboard
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('COVID-19 Global Key Findings Dashboard', fontsize=18, fontweight='bold')

# 1. Top 10 Countries by Cases
top10 = df_clean.sort_values(by='Cases', ascending=False).head(10)
sns.barplot(ax=axes[0, 0], data=top10, x='Cases', y='country', palette='Reds_r')
axes[0, 0].set_title('Top 10 Affected Countries', fontsize=12, fontweight='bold')
axes[0, 0].xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f'{x*1e-6:.0f}M'))

# 2. Cases Distribution by Continent
cont_cases = df_clean.groupby('continent')['Cases'].sum().reset_index().sort_values(by='Cases', ascending=False)
sns.barplot(ax=axes[0, 1], data=cont_cases, x='Cases', y='continent', palette='Blues_r')
axes[0, 1].set_title('Cases Distribution by Continent', fontsize=12, fontweight='bold')
axes[0, 1].xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f'{x*1e-6:.0f}M'))

# 3. Average Death Rate (%) by Continent
cont_rates = df_clean.groupby('continent')[['Cases', 'Deaths']].sum().reset_index()
cont_rates['Death_Rate'] = (cont_rates['Deaths'] / cont_rates['Cases']) * 100
cont_rates = cont_rates.sort_values(by='Death_Rate', ascending=False)
sns.barplot(ax=axes[1, 0], data=cont_rates, x='Death_Rate', y='continent', palette='Oranges_r')
axes[1, 0].set_title('Death Rate (%) by Continent', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Death Rate (%)')

# 4. Metric Correlation Heatmap
corr = df_clean[['population', 'Cases', 'Deaths', 'Recovered', 'Tests']].corr()
sns.heatmap(ax=axes[1, 1], data=corr, annot=True, cmap='coolwarm', fmt='.2f')
axes[1, 1].set_title('Metric Correlation Matrix', fontsize=12, fontweight='bold')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## 3. Machine Learning: Risk Category Classification

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Categorize Risk
def categorize_risk(rate):
    if rate >= 2.0:
        return 'High Risk'
    elif rate >= 1.0:
        return 'Medium Risk'
    else:
        return 'Low Risk'

df_clean['Risk_Category'] = df_clean['Death_Rate (%)'].apply(categorize_risk)

# Features and Target
features = ['population', 'Cases', 'Recovered', 'Tests', 'Death_Rate (%)']
X = df_clean[features]
y = df_clean['Risk_Category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print('Accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))